# Profilage Automatique des données

Ce notebook génère automatiquement des rapports de qualité des données en utilisant :
- **Evidently** : Rapport de qualité détaillé avec métriques avancées
- **Sweetviz** : Analyse exploratoire visuelle interactive

Les données sont chargées directement depuis la base PostgreSQL.

In [ ]:
# Import des bibliothèques
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
from evidently.legacy.report import Report
from evidently.legacy.metric_preset import DataQualityPreset
import sweetviz as sv
from datetime import datetime

In [ ]:
# Configuration de la connexion à la base de données
load_dotenv('../.env')  # Charger le fichier .env du répertoire parent

DB_HOST = os.getenv('DB_HOST', 'host.docker.internal')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('POSTGRES_DB', 'games_db')
DB_USER = os.getenv('POSTGRES_USER', 'postgres')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'postgres')

CONNECTION_STRING = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connexion à : {DB_HOST}:{DB_PORT}/{DB_NAME} avec utilisateur {DB_USER}")

In [ ]:
# Chargement des données depuis PostgreSQL
try:
    engine = create_engine(CONNECTION_STRING)
    
    # Charger toutes les données de la table games_brut
    df = pd.read_sql("SELECT * FROM games_brut", engine)
    
    print(f"✅ Données chargées avec succès !")
    print(f"📊 Shape : {df.shape[0]} lignes, {df.shape[1]} colonnes")
    print(f"🗄️ Colonnes : {list(df.columns)}")
    
    # Aperçu des données
    display(df.head())
    
except Exception as e:
    print(f"Erreur lors du chargement des données : {e}")
    df = None

In [ ]:
# Génération du rapport Evidently
if df is not None:
    try:
        print("🔍 Génération du rapport Evidently...")
        
        # Créer le rapport de qualité des données
        report = Report(metrics=[
            DataQualityPreset(),
        ])
        
        # Exécuter le rapport
        report.run(current_data=df, reference_data=None)
        
        # Sauvegarder avec timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        evidently_filename = f"../reports/evidently_data_quality_report_{timestamp}.html"
        report.save_html(evidently_filename)
        
        print(f"Evidently : {evidently_filename}")
        
    except Exception as e:
        print(f"Erreur Evidently : {e}")
else:
    print("Pas de données à analyser")

In [ ]:
# Génération du rapport Sweetviz
if df is not None:
    try:
        print("Génération du rapport Sweetviz")
        
        # Analyser les données avec Sweetviz
        sv_report = sv.analyze(df)
        
        # Sauvegarder avec timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        sweetviz_filename = f"../reports/sweetviz_report_{timestamp}.html"
        sv_report.show_html(sweetviz_filename, open_browser=False)
        
        print(f"Sweetviz : {sweetviz_filename}")
        
    except Exception as e:
        print(f"Erreur Sweetviz : {e}")
else:
    print("Pas de données à analyser")